# Mini-project · Your own province

**Day 3 · about 25 minutes · Student notebook**

> **Goal.** Produce one map and one defensible number about a province you care about, and be able to say where every figure came from.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## The brief

Pick **one province** and **one question**. Produce:

1. **One map** — with a legend, a title, and the province boundary drawn on it.
2. **One number** — in hectares, tonnes, or percent.
3. **One sentence** that states the number, the dataset, and the year.

You have about 25 minutes. Scope small and finish, rather than scope big and have nothing to show.

### Pick a question

| # | Question | Reuse from |
|---|---|---|
| 1 | How much of my province is forest, and how does that compare to a neighbour? | Lab 2 |
| 2 | How much of my province's forest is deciduous vs evergreen? | Lab 3 |
| 3 | How much carbon is stored in my province's forest? | Lab 4 |
| 4 | How much burned in my province in each of the last three years? | Lab 5 |
| 5 | Is my province's dry season getting drier over the last ten years? | Lab 6 |
| 6 | How much forest has my province lost since 2001? | new — see below |

### Your setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
# Study area: one Thai province. No shapefile upload needed.
PROVINCE = 'Nan'        # <-- change to your own province

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

area_ha = aoi.area(maxError=100).divide(1e4).getInfo()
print(f'{PROVINCE}: {area_ha:,.0f} hectares')

### If you chose question 6, here is the one dataset you have not used yet

In [ ]:
# Hansen Global Forest Change - the standard deforestation dataset.
# NOTE the version number: it goes up EVERY year. 2025_v1_13 is current as of August 2026.
gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')

# treecover2000 = % canopy cover in the year 2000
# loss          = 1 where forest was lost at some point
# lossyear      = 1..25, meaning 2001..2025

forest2000 = gfc.select('treecover2000').gte(30)      # "forest" = at least 30% canopy
loss = gfc.select('loss').And(forest2000)

lost_ha = (ee.Image.pixelArea().divide(1e4).updateMask(loss)
           .reduceRegion(ee.Reducer.sum(), aoi, 100,
                         maxPixels=1e9, bestEffort=True).getInfo()['area'])
print(f'Forest lost 2001-2025: {lost_ha:,.0f} ha')

> ### ✓ Check your answer
>
> For **Nan** this gives **109,087 ha** lost since 2001 — about 9% of the province, and roughly 11% of the forest that was standing in 2000. If you get a number larger than your province, you forgot the `treecover2000` threshold and are counting loss in areas that were never forest.

### Your work

In [ ]:
# Your analysis here.
# Copy the pattern from whichever lab matches your question -
# you are not expected to invent anything new.

In [ ]:
# Your map here. Do not forget: legend, boundary, sensible palette.

### Your one sentence

Fill this in. Every blank matters — the number alone is not an answer.

> In **______________** province, **______________** is **______________**,
> according to **______________** for the year(s) **______________**.

**Example, written properly:**

> In Nan province, forest covers 1,033,493 ha (84% of the province), according to
> ESA WorldCover v200 for the year 2021.

**The same claim, written badly:**

> Nan is 84.4% forest.

The second version cannot be checked, cannot be reproduced, and will be wrong the moment
someone uses a different product. Do not write the second version.

### Before you present, run the sniff test one last time

- [ ] Is the number in the units I claimed?
- [ ] Does the total add up to less than the province area?
- [ ] Would a second dataset give roughly the same answer?
- [ ] Can I say which dataset and which year, without looking it up?
- [ ] Does the map have a legend?